# Value iteration

_Artificial Intelligence (CS221)_

**Guess the values, improve them with the best action, repeat until they settle.**

We want the best value for each state, not the value of some given policy.

     Value iteration starts with rough guesses and improves them round by round.

---

This notebook is a practice scaffold for the **Value iteration** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np

# 3 states, 2 actions. T[a, s, s']; state 2 is the absorbing goal (reward 10).
T = np.array([
    [[0.8,0.2,0.0],[0.0,0.8,0.2],[0.0,0.0,1.0]],
    [[0.2,0.8,0.0],[0.0,0.2,0.8],[0.0,0.0,1.0]],
])
R = np.array([0.0, 0.0, 10.0])
gamma = 0.9
V = np.zeros(3)

for it in range(40):
    Q = T @ (R + gamma * V)              # Q[a, s]
    newV = Q.max(axis=0)                 # best action's value, per state
    if np.max(np.abs(newV - V)) < 1e-6:
        print("converged after", it, "iterations")
        break
    V = newV

policy = (T @ (R + gamma * V)).argmax(axis=0)
print("optimal values V*:", np.round(V, 3))
print("optimal policy   :", policy)

## Visualize the data & results

_On a real 3x4 warehouse floor with a goal shelf and a hazard spill, what is each cell worth and where should the robot go?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1,1), (0,3), (1,3)
gamma, step_cost = 0.9, -0.04
acts = {"up":(-1,0),"down":(1,0),"left":(0,-1),"right":(0,1)}
def ok(s): return 0<=s[0]<ROWS and 0<=s[1]<COLS and s != WALL
states = [(r,c) for r in range(ROWS) for c in range(COLS) if (r,c)!=WALL]
def reward(s): return 1.0 if s==GOAL else (-1.0 if s==HAZARD else step_cost)
def trans(s, a):                       # 0.8 intended, 0.1 each perpendicular
    perp = ["left","right"] if a in ("up","down") else ["up","down"]
    res = {}
    for p, mv in [(0.8,a),(0.1,perp[0]),(0.1,perp[1])]:
        d = acts[mv]; ns = (s[0]+d[0], s[1]+d[1])
        if not ok(ns): ns = s
        res[ns] = res.get(ns, 0) + p
    return res

V = {s: 0.0 for s in states}
for sweep in range(1000):
    nV, delta = {}, 0
    for s in states:
        if s in (GOAL, HAZARD): nV[s] = reward(s); continue
        nV[s] = reward(s) + gamma * max(
            sum(p*V[ns] for ns,p in trans(s,a).items()) for a in acts)
        delta = max(delta, abs(nV[s] - V[s]))
    V = nV
    if delta < 1e-8: break

grid = np.full((ROWS, COLS), np.nan)
for s in states: grid[s] = V[s]
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, cmap="viridis")
for s in states:
    ax.text(s[1], s[0], round(V[s], 2), ha="center", va="center", color="white")
ax.set_title("warehouse floor: optimal value V* per cell")
fig.colorbar(im, ax=ax); plt.show()